In [ ]:
import asyncio
import json
import os
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")


## దశ 1: నిర్మిత అవుట్పుట్స్ కోసం పిడాంటిక్ నమూనాలను నిర్వచించండి

ఈ నమూనాలు ఏజెంట్లు നൽകി తలుపు తిప్పే **స్కీమాను** నిర్వచిస్తాయి. పిడాంటిక్‌తో `response_format` ఉపయోగించడం నిర్ధారిస్తుంది:
- ✅ టైప్-సేఫ్ డేటా ఎక్స్‌ట్రాక్షన్
- ✅ ఆటోమేటిక్ వెరిఫికేషన్
- ✅ ఫ్రీ-టెక్ట్ రెస్పాన్స్ల నుండి పార్సింగ్ తప్పిదాలు లేవు
- ✅ ఫీల్డ్‌ల ఆధారంగా సులభమైన షరతుపల రూటింగ్


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## దశ 2: హోటల్ బుకింగ్ టూల్‌ను సృష్టించండి

ఈ టూల్ **availability_agent** కాల్ చేసి గదులు అందుబాటులో ఉన్నాయా అని చెక్ చేస్తుంది. మేము `@ai_function` డెకరేటర్‌ను ఉపయోగిస్తాము:
- ఒక Python ఫంక్షన్‌ను AI కాల్ చేయగల టూల్‌గా మార్చడానికి
- LLM కోసం ఆटोమేటిక్ గా JSON స్కీమాను రూపొందించడానికి
- ప్రామాణిక పరామితుల నిర్ధారణ చేయడానికి
- ఏజెంట్ల ద్వారా ఆటోమేటిక్ ఆహ్వానం సాధ్యమయ్యేలా చేయడానికి

ఈ డెమో కోసం:
- **స్టాక్‌హోమ్, సియాటిల్, టోక్యో, లండన్, అం‌స్టర్‌డామ్** → గదులు ఉన్నాయి ✅
- **ఇతర అన్ని నగరాలు** → గదులు లేవు ❌


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## దశ 3: రూటింగ్ కోసం షరతు ఫంక్షన్లను నిర్వచించండి

ఈ ఫంక్షన్లు ఏజెంట్ ప్రతిస్పందనను తనిఖీ చేస్తాయి మరియు వర్క్‌ఫ్లోలో ఏ దారిని తీసుకోవాలో నిర్ణయిస్తాయి.

**ముఖ్య నమూనా:**
1. సందేశం `AgentExecutorResponse` అని తనిఖీ చేయండి
2. సరైన అవుట్పుట్‌ను (Pydantic మోడల్) పార్స్ చేయండి
3. రూటింగ్‌ను నియంత్రించడానికి `True` లేదా `False` ను రిటర్న్ చేయండి

వర్క్‌ఫ్లో ఈ షరతులను **అంచುలు** పై అంచనా వేసి తదుపరి ఏ ఎగ్జిక్యూటర్‌ను ఆహ్వానించాలో నిర్ణయిస్తుంది.


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels ARE available.
    
    Returns True if the destination has hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return True  # Default to True if unexpected type

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels are NOT available.
    
    Returns True if the destination has no hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception as e:
        return False


print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")

## దశ 4: కస్టమ్ డిస్ప్లే ఎగ్జిక్యూటర్ సృష్టించండి

ఎగ్జిక్యూటర్లు ట్రాన్స్ఫర్మేషన్లు లేదా సైడ్ ఎఫెక్ట్‌లను నిర్వహించే వర్క్‌ఫ్లో భాగాలు. చివరి ఫలితాన్ని ప్రదర్శించడానికి కస్టమ్ ఎగ్జిక్యూటర్ సృష్టించడానికి మేము `@executor` డెకరేట్‌ను ఉపయోగిస్తాము.

**ముఖ్యమైన భావనలు:**
- `@executor(id="...")` - ఫంక్షన్‌ను వర్క్‌ఫ్లో ఎగ్జిక్యూటర్‌గా రిజిస్టర్ చేస్తుంది
- `WorkflowContext[Never, str]` - ఇన్పుట్/ఆుట్పుట్ కొరకు టైప్ సూచనలు
- `ctx.yield_output(...)` - చివరి వర్క్‌ఫ్లో ఫలితాన్ని ఇస్తుంది


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created with @executor decorator")

## దశ 5: ಪರಿಸರ చలామణులని లోడ్ చేయండి

LLM క్లయింట్‌ను కాన్ఫిగర్ చేయండి. ఈ ఉదాహరణ క్రింది వాటితో పనిచేస్తుంది:
- **Microsoft Foundry** / **Azure OpenAI** (ప్రతిస్పందనలు API — సిఫార్సు చేయబడింది)
- **Azure OpenAI**
- **OpenAI**


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Microsoft Foundry provider with keyless authentication
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


## దశ 6: నిర్మిత అవుట్పుట్లతో AI ఏజెంట్లు సృష్టించండి

మేము **మూడు ప్రత్యేక ఏజెంట్లను** సృష్టిస్తాము, ప్రతి ఒక్కటిని `AgentExecutor` లో చుట్టబెట్టాము:

1. **availability_agent** - టూల్ ఉపయోగించి హోటల్ అందుబాటు తనిఖీ చేస్తుంది
2. **alternative_agent** - ప్రత్యామ్నాయ పట్టణాలను సూచిస్తుంది (గదులు లేనప్పుడు)
3. **booking_agent** - బుకింగ్ కు ప్రోత్సహిస్తుంది (గదులు లభ్యమైతే)

**ముఖ్య లక్షణాలు:**
- `tools=[hotel_booking]` - ఏజెంటుకు టూల్ అందిస్తుంది
- `response_format=PydanticModel` - నిర్మిత JSON అవుట్పుట్ ను బలపరిచుతుంది
- `AgentExecutor(..., id="...")` - పని ప్రవాహం కోసం ఏజెంట్ ను చుట్టుకొస్తుంది


In [ ]:
# Agent 1: Check availability with tool
availability_agent = AgentExecutor(
    provider.as_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    provider.as_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    provider.as_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)


## దశ 7: షరతు ఆధారిత ఎడ్జిలతో వర్క్‌ఫ్లోని నిర్మించండి

ఇప్పుడు మనం `WorkflowBuilder` ను ఉపయోగించి షరతు ఆధారిత రూటింగ్ తో గ్రాఫ్ నిర్మించాలి:

**వర్క్‌ఫ్లో నిర్మాణం:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙         ↘
[no_availability]  [has_availability]
        ↓              ↓
alternative_agent  booking_agent
        ↓              ↓
    display_result ←───┘
```

**ప్రధాన పద్ధతులు:**
- `.set_start_executor(...)` - ప్రవేశ స్థానాన్ని సెట్ చేస్తుంది
- `.add_edge(from, to, condition=...)` - షరతు ఆధారిత ఎడ్జ్ జతచేస్తుంది
- `.build()` - వర్క్‌ఫ్లోని సిధ్ధం చేస్తుంది


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing:</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result
        </p>
    </div>
""")
)

## దశ 8: టెస్ట్ కేస్ 1 నడపండి - అందుబాటులోలేని నగరం (పారిస్)

మన సిమ్యులేషన్‌లో రూమ్‌లు లేని పారిస్‌లో హోటళ్లు కోరడం ద్వారా **అందుబాటులో లేకపోవడం** మార్గాన్ని పరీక్షిద్దాం.


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → alternative_agent → display_result</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run the workflow
events_paris = await workflow.run(request_paris)
outputs_paris = events_paris.get_outputs()

# Display results
if outputs_paris:
    result_paris = AlternativeResult.model_validate_json(outputs_paris[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT (Paris)</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_paris.alternative_destination}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_paris.reason}</p>
            </div>
        </div>
    """)
    )

## దశ 9: టెస్ట్ కేస్ 2 - అందుబాటుతో నగరం (స్టాక్‌హోమ్) నడపండి

ఇప్పుడు మన సిమ్యులేషన్‌లో గదులు ఉన్న స్టాక్‌హోమ్‌లో హోటల్స్‌ని అభ్యర్థించి **అందుబాటు** మార్గాన్ని పరీక్షిద్దాం.


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run the workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
            </div>
        </div>
    """)
    )

## కీలక బిందువులు మరియు తదుపరి చరణాలు

### ✅ మీరు నేర్చుకున్నది:

1. **WorkflowBuilder రీతిసౌకర్యం**
   - ప్రవేశ బిందువు నిర్వచించేందుకు `.set_start_executor()` ఉపయోగించండి
   - షరతు ఆధారిత రూటింగ్ కోసం `.add_edge(from, to, condition=...)` వాడండి
   - వర్క్‌ఫ్లోని చివరగా పూర్తి చేయడానికి `.build()` కాల్ చేయండి

2. **షరతుచేసిన రూటింగ్**
   - షరతు ఫంక్షన్లు `AgentExecutorResponse` ని పరిశీలిస్తాయి
   - రూటింగ్ నిర్ణయాలు తీసుకునేందుకు నిర్మాణాత్మక అవుట్‌పుట్‌లను పార్స్ చేయండి
   - ఒక ఎడ్జ్‌ను ప్రారంభించేందుకు `True` పంపండి, దాటివేయడానికి `False`

3. **టూల్ సమన్వయం**
   - Python ఫంక్షన్లను AI టూల్స్‌గా మార్చేందుకు `@ai_function` ఉపయోగించండి
   - అవసరమైతే ఏజెంట్లు టూల్స్‌ను ఆటోమేటిక్‌గా పిలుస్తాయి
   - టూల్స్ JSONని రిటర్న్ చేస్తాయి, ఏజెంట్లు దాన్ని పార్స్ చేస్తాయి

4. **నిర్మాణాత్మక అవుట్‌పుట్లు**
   - టైప్-సురక్షిత డేటా ఎక్స్‌ట్రాక్షన్ కోసం Pydantic మోడల్స్ వాడండి
   - ఏజెంట్లు సృష్టించినప్పుడు `response_format=MyModel` సెట్ చేయండి
   - `Model.model_validate_json()` తో ప్రత్యుత్తరాలను పార్స్ చేయండి

5. **అనుకూల ఎజిక్యూటర్లు**
   - వర్క్‌ఫ్లో భాగాలు సృష్టించడానికి `@executor(id="...")` ఉపయోగించండి
   - ఎజిక్యూటర్లు డేటా మార్చవచ్చు లేదా పక్కటి ప్రభావాలు చేయవచ్చు
   - వర్క్‌ఫ్లో ఫలితాలు ఉత్పత్తి చేయడానికి `ctx.yield_output()` వాడండి

### 🚀 నిజ జీవిత అనువర్తనాలు:

- **యాత్ర బుకింగ్**: అందుబాటును తనిఖీ చేయడం, ప్రత్యామ్నాయాలు సూచించడం, ఎంపికలను పోల్చడం
- **కస్టమర్ సర్వీస్**: సమస్య రకం, భావోద్వేగం, ప్రాధాన్యత ఆధారంగా రూటింగ్
- **ఈ-కామర్స్**: ఇన్వెంటరీ తనిఖీ, ప్రత్యామ్నాయాలు సూచించడం, ఆర్డర్లు ప్రాసెస్ చేయడం
- **కంటెంట్ మోడరేషన్లు**: విషపూరిత స్కోర్లు, వాడుకరి ఫ్లాగ్‌ల ఆధారంగా రూటింగ్
- **అనుమతి వర్క్‌ఫ్లోలు**: సొమ్ము, వాడుకరి పాత్ర, ప్రమాదస్థాయి ఆధారంగా రూటింగ్
- **బహుళ దశల ప్రాసెసింగ్**: డేటా నాణ్యత, పూర్తి స్థాయిని బట్టి రూటింగ్

### 📚 తదుపరి చరణాలు:

- మరింత సంక్లిష్ట షరతులు (బహుళ ప్రమాణాలు) జోడించండి
- వర్క్‌ఫ్లో స్థితి నిర్వహణతో లూప్‌లు అమలు చేయండి
- పునర్వినియోగపరచుకొనగల భాగాల కోసం ఉప-వర్క్‌ఫ్లోలను జోడించండి
- నిజమైన APIలతో సమన్వయం చేయండి (హోటల్ బుకింగ్, ఇన్వెంటరీ వ్యవస్థలు)
- లోప నిర్వహణ మరియు ప్రత్యామ్నాయ మార్గాలను జోడించండి
- నిర్మిత దృశ్యీకరణ సాధనాలతో వర్క్‌ఫ్లోలను కாட்சీకరించండి


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్వీకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలు తప్పులు లేదా అసమగ్రతలను కలిగి ఉండవచ్చు. దాని స్వదేశ భాషలో ఉన్న అసలు పత్రాన్ని అధికారం కలిగిన మూలంగా పరిగణించాలి. కీలకమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదాన్ని సిఫారసు చేస్తాము. ఈ అనువాదం ఉపయోగం వల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పుదారులు కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
